In [6]:
import os
import glob
from vip_slap2_analysis.io.matv73 import MatV73File
from vip_slap2_analysis.glutamate.summary import GlutamateSummary

In [2]:
datapath = r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-25_803496\803496_2025-07-25_13-02-10_slap2_2026-01-18_05-53-08\source_extraction\ExperimentSummary\SummaryLoCo-260117-185357.mat"

In [34]:
from pathlib import Path
import h5py
import numpy as np

def _safe_keys(obj):
    try:
        return list(obj.keys())
    except Exception:
        return []

def _decode_h5_string(x):
    if isinstance(x, bytes):
        try:
            return x.decode("utf-8")
        except Exception:
            return repr(x)
    if isinstance(x, np.ndarray) and x.dtype.kind in {"S", "U"}:
        return x.astype(str).tolist()
    return x

def _describe_node(node, max_children=20):
    if isinstance(node, h5py.Dataset):
        desc = {
            "kind": "dataset",
            "shape": tuple(node.shape),
            "dtype": str(node.dtype),
        }
        try:
            if node.shape == ():
                desc["scalar_value"] = _decode_h5_string(node[()])
            elif node.size <= 10:
                desc["sample_value"] = _decode_h5_string(node[()])
        except Exception:
            pass
        return desc

    if isinstance(node, h5py.Group):
        keys = _safe_keys(node)
        return {
            "kind": "group",
            "n_children": len(keys),
            "children_preview": keys[:max_children],
        }

    return {"kind": str(type(node))}

def _resolve_ref(file_handle, ref):
    """
    Resolve an HDF5 object reference to the referenced object.
    """
    try:
        return file_handle[ref]
    except Exception:
        return None

def _try_get_E(exp):
    """
    Return exp['E'] if present, otherwise raise a helpful error.
    """
    if "E" in exp:
        return exp["E"]
    raise KeyError(
        f"'E' not found under exptSummary. Available exptSummary keys: {_safe_keys(exp)}"
    )

def inspect_summary_mat_structure(summary_mat_path, dmd=1, trial=1, verbose=True):
    """
    Inspect one SummaryLoCo .mat file.

    This version assumes the common layout:
        /exptSummary/E

    It handles MATLAB v7.3 cell arrays stored as HDF5 object refs.
    """
    summary_mat_path = Path(summary_mat_path)

    out = {
        "path": str(summary_mat_path),
        "top_level_keys": None,
        "exptSummary_keys": None,
        "E_shape": None,
        "E_dtype": None,
        "representative_index_used": None,
        "representative_group_path": None,
        "representative_group_keys": None,
        "signals": {},
    }

    with h5py.File(summary_mat_path, "r") as f:
        out["top_level_keys"] = _safe_keys(f)

        if "exptSummary" not in f:
            raise KeyError(
                f"No 'exptSummary' group found. Top-level keys: {out['top_level_keys']}"
            )

        exp = f["exptSummary"]
        out["exptSummary_keys"] = _safe_keys(exp)

        E = _try_get_E(exp)
        out["E_shape"] = tuple(E.shape)
        out["E_dtype"] = str(E.dtype)

        dmd0 = dmd - 1
        trial0 = trial - 1

        # Some files store E as (n_dmd, n_trials), some as (n_trials, n_dmd)
        candidate_indices = [(dmd0, trial0), (trial0, dmd0)]

        ref_obj = None
        ref_idx = None

        for idx in candidate_indices:
            try:
                ref = E[idx]
                obj = _resolve_ref(f, ref)
                if obj is not None:
                    ref_obj = obj
                    ref_idx = idx
                    break
            except Exception:
                continue

        if ref_obj is None:
            raise ValueError(
                f"Could not resolve a valid E reference for dmd={dmd}, trial={trial}. "
                f"E shape={out['E_shape']}, dtype={out['E_dtype']}"
            )

        out["representative_index_used"] = ref_idx
        out["representative_group_path"] = ref_obj.name
        out["representative_group_keys"] = _safe_keys(ref_obj)

        for key in _safe_keys(ref_obj):
            node = ref_obj[key]
            if isinstance(node, h5py.Group):
                children = {}
                for subkey in _safe_keys(node):
                    children[subkey] = _describe_node(node[subkey])
                out["signals"][key] = {
                    "kind": "group",
                    "children": children,
                }
            else:
                out["signals"][key] = _describe_node(node)

    if verbose:
        print_summary_structure(out)

    return out

def print_summary_structure(info):
    print("=" * 80)
    print("PATH")
    print(info["path"])
    print("\nTOP-LEVEL KEYS")
    print(info["top_level_keys"])
    print("\nexptSummary KEYS")
    print(info["exptSummary_keys"])
    print("\nE")
    print(f"  shape: {info['E_shape']}")
    print(f"  dtype: {info['E_dtype']}")
    print(f"  representative index used: {info['representative_index_used']}")
    print(f"  representative group path: {info['representative_group_path']}")
    print(f"  representative group keys: {info['representative_group_keys']}")
    print("\nREPRESENTATIVE SIGNALS")
    for key, spec in info["signals"].items():
        print(f"\n- {key}")
        if spec["kind"] == "group":
            for subkey, subspec in spec["children"].items():
                print(f"    {subkey}: {subspec}")
        else:
            print(f"    {spec}")
    print("=" * 80)

In [38]:
def print_trace_relevant_fields(summary_mat_path, dmd=1, trial=1):
    with h5py.File(summary_mat_path, "r") as f:
        exp = f["exptSummary"]
        E = exp["E"]

        dmd0 = dmd - 1
        trial0 = trial - 1

        obj = None
        idx_used = None
        for idx in [(dmd0, trial0), (trial0, dmd0)]:
            try:
                candidate = f[E[idx]]
                obj = candidate
                idx_used = idx
                break
            except Exception:
                continue

        if obj is None:
            raise ValueError("Could not resolve representative E entry.")

        print(f"Resolved E index: {idx_used}")
        print(f"Group path: {obj.name}")
        print(f"Top-level keys in representative trial group: {list(obj.keys())}")
        print()

        for key in obj.keys():
            node = obj[key]
            if isinstance(node, h5py.Dataset):
                print(f"{key}: dataset shape={node.shape}, dtype={node.dtype}")
            elif isinstance(node, h5py.Group):
                print(f"{key}: group")
                for subkey in node.keys():
                    subnode = node[subkey]
                    if isinstance(subnode, h5py.Dataset):
                        print(f"  - {subkey}: shape={subnode.shape}, dtype={subnode.dtype}")
                    else:
                        print(f"  - {subkey}: type={type(subnode)}")
            else:
                print(f"{key}: type={type(node)}")

In [36]:
info = inspect_summary_mat_structure(datapath, dmd=1, trial=1)
info

PATH
\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-25_803496\803496_2025-07-25_13-02-10_slap2_2026-01-18_05-53-08\source_extraction\ExperimentSummary\SummaryLoCo-260117-185357.mat

TOP-LEVEL KEYS
['#refs#', 'exptSummary']

exptSummary KEYS
['E', 'Z', 'aData', 'actIM', 'dr', 'meanIM', 'params', 'peaks', 'perTrialActIMsAligned', 'perTrialActIms', 'perTrialAlignmentOffsets', 'perTrialMeanIMs', 'perTrialMeanIMsAligned', 'selPix', 'trialTable', 'userROIs']

E
  shape: (2, 62)
  dtype: object
  representative index used: (0, 0)
  representative group path: /#refs#/f
  representative group keys: ['F0', 'ROIs', 'SNR', 'dF', 'discardFrames', 'footprints', 'global']

REPRESENTATIVE SIGNALS

- F0
    {'kind': 'dataset', 'shape': (5613, 92), 'dtype': 'float64'}

- ROIs
    F: {'kind': 'dataset', 'shape': (8, 1, 1), 'dtype': 'float64', 'sample_value': array([[[nan]],

       [[nan]],

       [[nan]],

       [[nan]],

       [[nan]],

       [[nan]],

       [[na

{'path': '\\\\allen\\aind\\scratch\\ophys\\Andrew\\VIP_synaptic_dynamics\\iGluSnFR4f\\803496\\2025-07-25_803496\\803496_2025-07-25_13-02-10_slap2_2026-01-18_05-53-08\\source_extraction\\ExperimentSummary\\SummaryLoCo-260117-185357.mat',
 'top_level_keys': ['#refs#', 'exptSummary'],
 'exptSummary_keys': ['E',
  'Z',
  'aData',
  'actIM',
  'dr',
  'meanIM',
  'params',
  'peaks',
  'perTrialActIMsAligned',
  'perTrialActIms',
  'perTrialAlignmentOffsets',
  'perTrialMeanIMs',
  'perTrialMeanIMsAligned',
  'selPix',
  'trialTable',
  'userROIs'],
 'E_shape': (2, 62),
 'E_dtype': 'object',
 'representative_index_used': (0, 0),
 'representative_group_path': '/#refs#/f',
 'representative_group_keys': ['F0',
  'ROIs',
  'SNR',
  'dF',
  'discardFrames',
  'footprints',
  'global'],
 'signals': {'F0': {'kind': 'dataset',
   'shape': (5613, 92),
   'dtype': 'float64'},
  'ROIs': {'kind': 'group',
   'children': {'F': {'kind': 'dataset',
     'shape': (8, 1, 1),
     'dtype': 'float64',
     's

In [40]:
print_trace_relevant_fields(datapath)

Resolved E index: (0, 0)
Group path: /#refs#/f
Top-level keys in representative trial group: ['F0', 'ROIs', 'SNR', 'dF', 'discardFrames', 'footprints', 'global']

F0: dataset shape=(5613, 92), dtype=float64
ROIs: group
  - F: shape=(8, 1, 1), dtype=float64
  - Fsvd: shape=(3,), dtype=uint64
SNR: dataset shape=(1, 92), dtype=float64
dF: group
  - denoised: shape=(5613, 92), dtype=float64
  - events: shape=(5613, 92), dtype=float64
  - ls: shape=(5613, 92), dtype=float64
discardFrames: dataset shape=(5613, 1), dtype=uint8
footprints: dataset shape=(92, 6614), dtype=float32
global: group
  - F: shape=(5613, 1), dtype=float64


In [16]:
from vip_slap2_analysis.glutamate.summary import GlutamateSummary

def probe_glutamate_trace_calls(summary_mat_path, dmd=1, trial=1):
    exp = GlutamateSummary(summary_mat_path)

    tests = [
        ("dF", "ls"),
        ("dF", "denoised"),
        ("dF", "events"),
        ("dFF", "ls"),
        ("dFF", "denoised"),
        ("dFF", "events"),
        ("F0", None),
    ]

    results = []

    for signal, mode in tests:
        try:
            if signal == "F0":
                x = exp.get_traces(
                    dmd=dmd,
                    trial=trial,
                    signal="F0",
                    squeeze_channels=False,
                )
            else:
                x = exp.get_traces(
                    dmd=dmd,
                    trial=trial,
                    signal=signal,
                    mode=mode,
                    squeeze_channels=False,
                )

            results.append({
                "signal": signal,
                "mode": mode,
                "ok": True,
                "shape": tuple(np.asarray(x).shape),
            })

        except Exception as e:
            results.append({
                "signal": signal,
                "mode": mode,
                "ok": False,
                "error": f"{type(e).__name__}: {e}",
            })

    return results

In [17]:
probe_glutamate_trace_calls(datapath, dmd=1, trial=1)

[{'signal': 'dF', 'mode': 'ls', 'ok': True, 'shape': (5613, 92, 1)},
 {'signal': 'dF', 'mode': 'denoised', 'ok': True, 'shape': (5613, 92, 1)},
 {'signal': 'dF', 'mode': 'events', 'ok': True, 'shape': (5613, 92, 1)},
 {'signal': 'dFF',
  'mode': 'ls',
  'ok': False,
  'error': 'KeyError: "\'dFF\' not found for dmd=1, trial=1. Available top-level keys: [\'F0\', \'ROIs\', \'SNR\', \'dF\', \'discardFrames\', \'footprints\', \'global\']"'},
 {'signal': 'dFF',
  'mode': 'denoised',
  'ok': False,
  'error': 'KeyError: "\'dFF\' not found for dmd=1, trial=1. Available top-level keys: [\'F0\', \'ROIs\', \'SNR\', \'dF\', \'discardFrames\', \'footprints\', \'global\']"'},
 {'signal': 'dFF',
  'mode': 'events',
  'ok': False,
  'error': 'KeyError: "\'dFF\' not found for dmd=1, trial=1. Available top-level keys: [\'F0\', \'ROIs\', \'SNR\', \'dF\', \'discardFrames\', \'footprints\', \'global\']"'},
 {'signal': 'F0', 'mode': None, 'ok': True, 'shape': (5613, 92, 1)}]

In [20]:
with h5py.File(Path(datapath), "r") as f:
    data = _safe_keys(f)